# K-Nearest Neighbors (KNN) Regressor

## AI/ML Engineer Training Program

**Notebook:** 06_KNN_Regressor.ipynb
**Phase:** Regression Models
**Objective:** Learn KNN Regressor and understand how it differs from KNN Classifier while following an industry-standard machine learning workflow.

### Learning Goals

By the end of this notebook, I should be able to:

* Explain the difference between KNN Classifier and KNN Regressor.
* Understand how KNN performs regression using neighboring values.
* Build a KNN regression pipeline with feature scaling.
* Tune `n_neighbors` using GridSearchCV.
* Evaluate regression models using MAE, MSE, RMSE, and R².
* Explain KNN Regressor in a machine learning interview.

### Workflow

1. Problem Statement
2. Import Libraries
3. Load Dataset
4. Exploratory Data Analysis (EDA)
5. Missing Values & Duplicates
6. Train-Test Split
7. KNN Regression Pipeline
8. Baseline Model
9. Model Evaluation
10. Hyperparameter Tuning
11. Tuned Model Evaluation
12. Professional Summary
13. Business Decision


## 1. Problem Statement

The objective of this project is to build a **K-Nearest Neighbors Regressor** that predicts **California house prices** using numerical housing features.

Unlike KNN Classifier, which predicts categories, **KNN Regressor predicts continuous numerical values** by averaging the target values of the nearest neighbors.

This is a **regression problem** because the target variable is a continuous number (house price).


In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [3]:
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name="target")

print("X shape",X.shape)
print("y shape",y.shape)
X.head()

X shape (20640, 8)
y shape (20640,)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [4]:
print("Missing values:\n", X.isnull().sum())
print("\nDuplicate rows:", X.duplicated().sum())
print("\nTarget summary:\n", y.describe())

Missing values:
 MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int64

Duplicate rows: 0

Target summary:
 count    20640.000000
mean         2.068558
std          1.153956
min          0.149990
25%          1.196000
50%          1.797000
75%          2.647250
max          5.000010
Name: target, dtype: float64


In [8]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [12]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsRegressor(n_neighbors=5))
])

knn_pipeline.fit(X_train, y_train)

y_pred = knn_pipeline.predict(X_test)

In [13]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:.4f}")
print(f"MSE : {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

MAE : 0.4462
MSE : 0.4324
RMSE: 0.6576
R²  : 0.6700


### Baseline KNN Regressor Performance

| Metric |      Value |
| ------ | ---------: |
| MAE    | **0.4462** |
| MSE    | **0.4324** |
| RMSE   | **0.6576** |
| R²     | **0.6700** |

### Engineering Interpretation

The baseline KNN Regressor explains **67% of the variance in California house prices**.

The average prediction error is approximately **0.446 housing units**, which corresponds to about **$44,620** because the target variable is measured in units of **$100,000**.

This provides a strong baseline for subsequent hyperparameter tuning.


In [17]:
param_grid = {
    "model__n_neighbors": list(range(1, 31))
}

grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV R²:", grid_search.best_score_)

Best Parameters: {'model__n_neighbors': 9}
Best CV R²: 0.6921550952979508


### Hyperparameter Tuning Results

GridSearchCV identified **`n_neighbors = 9`** as the optimal value.

The best cross-validation **R² score was 0.6922**, indicating that the tuned model is expected to generalize better than the baseline configuration.

A slightly larger neighborhood produced a more stable regression model by reducing sensitivity to noisy observations.


In [18]:
best_model = grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_tuned)
mse = mean_squared_error(y_test, y_pred_tuned)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_tuned)

print(f"MAE : {mae:.4f}")
print(f"MSE : {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

MAE : 0.4409
MSE : 0.4246
RMSE: 0.6516
R²  : 0.6760


## Final Model Comparison

| Metric | Baseline (K=5) | Tuned (K=9) |
| ------ | -------------: | ----------: |
| MAE    |     **0.4462** |  **0.4409** |
| MSE    |     **0.4324** |  **0.4246** |
| RMSE   |     **0.6576** |  **0.6516** |
| R²     |     **0.6700** |  **0.6760** |

### Engineering Interpretation

Hyperparameter tuning improved all evaluation metrics.

The tuned model with **K = 9** achieved a lower prediction error and a higher **R² score**, indicating better generalization performance than the baseline model.

### Business Decision

The tuned KNN Regressor is preferred for deployment because it provides more accurate house price predictions while maintaining a simple and interpretable modeling approach.


In [19]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

# Fetch dataset
concrete = fetch_ucirepo(id=165)

# Features and target
X = concrete.data.features
y = concrete.data.targets.squeeze()

print(X.shape)
print(y.shape)

X.head()

(1030, 8)
(1030,)


,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360


In [20]:
y.head()

0    79.99
1    61.89
2    40.27
3    41.05
4    44.30
Name: Concrete compressive strength, dtype: float64

In [22]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [23]:
print("x shape",X.shape)
print("y shape",y.shape)

x shape (1030, 8)
y shape (1030,)


In [24]:
knn_model = Pipeline([
    ("scalar",StandardScaler()),
    ("model",KNeighborsRegressor(n_neighbors=3))
])

In [25]:
knn_model.fit(X_train,y_train)

Pipeline(steps=[('scalar', StandardScaler()),
                ('model', KNeighborsRegressor(n_neighbors=3))])

In [26]:
y_pred_knn_base = knn_model.predict(X_test)

In [27]:
print("mae score:",mean_absolute_error(y_test,y_pred_knn_base))
print("mse score :",mean_squared_error(y_test,y_pred_knn_base))
print("rmse score: ",np.sqrt(mean_squared_error(y_test,y_pred_knn_base)))
print("r2 score :", r2_score(y_test,y_pred_knn_base))


mae score: 6.356504854368932
mse score : 66.10873193096008
rmse score:  8.130727638468779
r2 score : 0.7434433068036008


In [31]:
prama_grid_tune = {
    "model__n_neighbors":list(range(5,36))
}

In [37]:
param_grid = {
    "model__n_neighbors": list(range(1, 31))
}

grid_search = GridSearchCV(
    estimator=knn_model,      # your concrete pipeline
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scalar', StandardScaler()),
                                       ('model',
                                        KNeighborsRegressor(n_neighbors=3))]),
             n_jobs=-1,
             param_grid={'model__n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
                                                11, 12, 13, 14, 15, 16, 17, 18,
                                                19, 20, 21, 22, 23, 24, 25, 26,
                                                27, 28, 29, 30]},
             scoring='r2')

In [40]:
best_model_knn = grid_search.best_estimator_

y_pred_knn = best_model_knn.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_knn)
mse = mean_squared_error(y_test, y_pred_knn)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_knn)

print(f"MAE : {mae:.4f}")
print(f"MSE : {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

MAE : 6.7765
MSE : 74.2621
RMSE: 8.6175
R²  : 0.7118


In [41]:
print(grid_search.best_params_)
print(grid_search.best_score_)

{'model__n_neighbors': 4}
0.7066822563920903
